In [8]:
import sys
sys.path.append("../")  # go up one level from notebooks/ to project root (for using src/)
import os
from dotenv import load_dotenv
import json
import requests
from src.query_builder import build_semantic_scholar_query, load_keywords
from src.data_collection import collect_semantic_scholar_data, load_config

In [ ]:
# Load API keys from .env file
load_dotenv("../.env")
semantic_scholar_api_key = os.getenv("SEMANTIC_SCHOLAR_API_KEY")

# Semantic Scholar

# Didactic example
Didactic example of how to use the bulk search endpoint. 

In [4]:
# Semantic Scholar didactic example of how to use the bulk search endpoint

# Define the bulk search endpoint
url = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"

# Set up your query parameters
query_params = {
    "query": '"climate change" AND "social media" AND (misinformation OR polarization)',
    # "year": "2020",
    "publicationDateOrYear": "2020-01-01:2020-12-31", 
    "fields": "paperId,title,year"
    # "fields": "paperId,externalIds,title,year,publicationDate,abstract,publicationVenue,journal,citationCount,s2FieldsOfStudy,publicationTypes,authors"
}

# Optional: Add your API key if you have one to avoid rate limits
headers = {"x-api-key": semantic_scholar_api_key}

try:
    response = requests.get(url, params=query_params, headers=headers)
    response.raise_for_status()

    # Parse and print the JSON response
    data = response.json()
    print(json.dumps(data, indent=2))

except requests.exceptions.HTTPError as err:
    print(f"HTTP Error: {err}")
except Exception as e:
    print(f"An error occurred: {e}")

{
  "total": 2,
  "token": null,
  "data": [
    {
      "paperId": "0978a91dbbfdf0b9762fc572e9f7ff3b73b25add",
      "title": "Partisanship, Impeachment, and the Democratic Primaries: American Political Discourse, January - February 2020",
      "year": 2020
    },
    {
      "paperId": "85f4d04076aefafe60a8ead0435b34bf7d8fbfdd",
      "title": "Online misinformation about climate change",
      "year": 2020
    }
  ]
}


## Extract Semantic Scholar data

In [ ]:
config = load_config("../config.yml")

ss_config = config["semantic_scholar"]
date_start = config["search"]["date_start"]
date_end = config["search"]["date_end"]

In [5]:
# Build the query

logic = "climate AND nlp AND social_media AND (misinformation OR polarization)"

semantic_scholar_query = build_semantic_scholar_query(
    logic_string=logic,
    keywords_path="../data/search_query/keywords.yml"
)

print(semantic_scholar_query)
print("\nNumber of characters", len(semantic_scholar_query))
print("Semantic Scholar query length limit is 2000 characters") 

("global warming" | "climate change" | "climate science" | "climate denial" | "climate crisis" | "climate skeptic" | "climate skepticism" | "climate skeptics" | "climate skepticist" | "climate action" | "climate misinformation" | "climate disinformation") + ("natural language processing" | NLP | "machine learning" | LLM | AI | "artificial intelligence" | "language model" | "language models" | "neural network" | "neural networks" | GenAI | "Generative AI" | "text mining" | "text processing" | "text classification" | "deep learning" | "transformer*" | "argument mining" | "sentiment analysis") + ("social media" | "social network" | "social networks" | "social platform" | "social platforms" | "online community" | "online communities" | "alternative platform" | "alternative platforms" | alt-tech | Youtube | Rumble | Bitchute | Odysee | Twitch | Quora | Facebook | Instagram | Tiktok | Whatsapp | Telegram | Signal | WeChat | Discord | Threads | Mastodon | Reddit | X | Twitter/x | Twitter | Bl

In [ ]:
collect_semantic_scholar_data(
    query=semantic_scholar_query,
    fields=ss_config["fields_full"],
    output_path=f"../data/raw/semantic_scholar/semantic_scholar_data.jsonl",
    date_start=date_start,
    date_end=date_end,
    api_key=semantic_scholar_api_key
)

{'query': '("global warming" | "climate change" | "climate science" | "climate denial" | "climate crisis" | "climate skeptic" | "climate skepticism" | "climate skeptics" | "climate skepticist" | "climate action" | "climate misinformation" | "climate disinformation") + ("natural language processing" | NLP | "machine learning" | LLM | AI | "artificial intelligence" | "language model" | "language models" | "neural network" | "neural networks" | GenAI | "Generative AI" | "text mining" | "text processing" | "text classification" | "deep learning" | "transformer*" | "argument mining" | "sentiment analysis") + ("social media" | "social network" | "social networks" | "social platform" | "social platforms" | "online community" | "online communities" | "alternative platform" | "alternative platforms" | alt-tech | Youtube | Rumble | Bitchute | Odysee | Twitch | Quora | Facebook | Instagram | Tiktok | Whatsapp | Telegram | Signal | WeChat | Discord | Threads | Mastodon | Reddit | X | Twitter/x | T

88

## Examine the interseccions

In [ ]:
from itertools import combinations, chain

sets = ["climate", "nlp", "social_media", "misinformation", "polarization"]
all_intersections = list(chain.from_iterable(combinations(sets, r) for r in range(1, len(sets) + 1)))

for intersection in all_intersections:

    # Logic
    logic = " AND ".join(intersection)

    #Name for the file
    name = "_".join(logic.split(" "))
    
    # Build the query
    semantic_scholar_query = build_semantic_scholar_query(
        logic_string=logic,
        keywords_path="../data/search_query/keywords.yml"
    )

    # Extract data
    collect_semantic_scholar_data(
        query=semantic_scholar_query,
        fields=ss_config["fields_lite"],
        output_path=f"../data/raw/semantic_scholar/sets/{name}.jsonl", 
        date_start=date_start,
        date_end=date_end,
        api_key=semantic_scholar_api_key
    )

climate
climate
nlp
nlp
social_media
social_media
misinformation
misinformation
polarization
polarization
climate AND nlp
climate_AND_nlp
climate AND social_media
climate_AND_social_media
climate AND misinformation
climate_AND_misinformation
climate AND polarization
climate_AND_polarization
nlp AND social_media
nlp_AND_social_media
nlp AND misinformation
nlp_AND_misinformation
nlp AND polarization
nlp_AND_polarization
social_media AND misinformation
social_media_AND_misinformation
social_media AND polarization
social_media_AND_polarization
misinformation AND polarization
misinformation_AND_polarization
climate AND nlp AND social_media
climate_AND_nlp_AND_social_media
climate AND nlp AND misinformation
climate_AND_nlp_AND_misinformation
climate AND nlp AND polarization
climate_AND_nlp_AND_polarization
climate AND social_media AND misinformation
climate_AND_social_media_AND_misinformation
climate AND social_media AND polarization
climate_AND_social_media_AND_polarization
climate AND misi

# Semantic Scholar 

In [ ]:
import json
import time
import requests
import os
import yaml
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

# ─────────────────────────────────────────────
# INTERSECTION EXPLORATION
# ─────────────────────────────────────────────

def run_intersection_queries(groups, logic_combinations, fields,
                              output_dir, date_start, date_end,
                              api_key="", base_url=None):
    """
    Run a series of intersection queries to understand the
    size and overlap of keyword groups.

    Args:
        groups:             dict of group_name -> formatted SS query clause
        logic_combinations: list of logic strings to run
                            e.g. ["climate", "nlp", "climate AND nlp"]
        fields:             list of fields (use fields_lite for this)
        output_dir:         directory to write results
        date_start:         start date
        date_end:           end date
        api_key:            optional API key
        base_url:           API endpoint
    """
    # from src.query.builder import build_query, load_keywords, to_semantic_scholar

    output_dir = Path(output_dir)
    date_str = datetime.now().strftime("%Y%m%d")
    summary = []

    for logic in logic_combinations:
        # build generic query then convert to SS syntax
        generic = build_query(logic, groups)
        ss_query = to_semantic_scholar(generic)

        # use logic string as filename (sanitize it)
        safe_name = logic.replace(" ", "_").replace("(", "").replace(")", "")
        output_path = output_dir / f"{date_str}_{safe_name}.jsonl"

        total = collect_semantic_scholar_data(
            query=ss_query,
            fields=fields,
            output_path=output_path,
            date_start=date_start,
            date_end=date_end,
            api_key=api_key,
            base_url=base_url
        )

        summary.append({"logic": logic, "total": total, "file": str(output_path)})

    # save summary
    summary_path = output_dir / f"{date_str}_intersection_summary.json"
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)

    print(f"Intersection summary saved to {summary_path}")
    return summary

In [61]:
# logic = "climate"
# semantic_scholar_query = build_semantic_scholar_query(
#     logic_string=logic,
#     keywords_path="../data/search_query/keywords.yml"
# )
# print(f"Query: {semantic_scholar_query}")

config = load_config("../config.yml")
api_key = load_api_key("../.env")

ss_config = config["semantic_scholar"]
date_start = config["search"]["date_start"]
date_end = config["search"]["date_end"]

In [64]:
collect_semantic_scholar_data(
    query=semantic_scholar_query,
    fields=ss_config["fields_full"],
    output_path=f"../data/raw/semantic_scholar/sets/climate.jsonl",
    date_start=date_start,
    date_end=date_end,
    api_key=api_key
)

{'query': '("global warming" | "climate change" | "climate science" | "climate denial" | "climate crisis" | "climate skeptic" | "climate skepticism" | "climate skeptics" | "climate skepticist" | "climate action" | "climate misinformation" | "climate disinformation") + ("natural language processing" | NLP | "machine learning" | LLM | AI | "artificial intelligence" | "language model" | "language models" | "neural network" | "neural networks" | GenAI | "Generative AI" | "text mining" | "text processing" | "text classification" | "deep learning" | "transformer*" | "argument mining" | "sentiment analysis") + ("social media" | "social network" | "social networks" | "social platform" | "social platforms" | "online community" | "online communities" | "alternative platform" | "alternative platforms" | alt-tech | Youtube | Rumble | Bitchute | Odysee | Twitch | Quora | Facebook | Instagram | Tiktok | Whatsapp | Telegram | Signal | WeChat | Discord | Threads | Mastodon | Reddit | X | Twitter/x | T

88